In [1]:
import pandas as pd
import numpy as np
from glob import glob
from os.path import join
from multiprocessing import Pool
import multiprocessing
import time
import pyarrow as pa
import pyarrow.parquet as pq
from skmob.core.trajectorydataframe import TrajDataFrame
from skmob.preprocessing import detection
from skmob.measures import individual

## Example Analysis Using Multiprocessing to Get Metrics for Users across time (all data for 2023)

This notebook is benchmarked using an single node on NCAR's Derecho machine with 128 CPUs and 256 total threads.

* **base_path**: Path to chunked (using parquet row groups) daily Veresat data.

* **key_file**: Path to the master key file which includes all user ID's for all of 2023.

* **daily_files**: List of all the chunked daily priocesses files.

See the spreasheet for actual paths.

In [35]:
base_path = ""
key_file = ""
daily_files = sorted(glob(join(base_path, "veraset*")))

Load a single file using the *pyarrow* package to inspect the data as an example.

Look at the schema and metadata.

In [7]:
p = pq.ParquetFile(daily_files[11])

In [8]:
p.schema

required group field_id=-1 schema {
  optional int64 field_id=-1 datetime (Timestamp(isAdjustedToUTC=true, timeUnit=nanoseconds, is_from_converted_type=false, force_set_converted_type=false));
  optional binary field_id=-1 id_type (String);
  optional float field_id=-1 latitude;
  optional float field_id=-1 longitude;
  optional int32 field_id=-1 horizontal_accuracy (Int(bitWidth=16, isSigned=true));
  optional binary field_id=-1 caid (String);
}

In [9]:
p.metadata

  created_by: parquet-cpp-arrow version 18.0.0-SNAPSHOT
  num_columns: 6
  num_rows: 79547343
  num_row_groups: 57431
  format_version: 2.6
  serialized_size: 39278638

We see that the data for this day has about 80 million rows and about 57 thousand "row groups", which are chunks that can be loaded individually.  

It is chunked by *1000 unique user IDs (caid)*. Since there are arbritray number of time steps per user id, each row group of data is likely to have a differnt amount of rows in it.

We can use the same package to load specific row groups of data from a file.

In [13]:
## Only pull row group 15
p.read_row_group(15).to_pandas()

,datetime,id_type,latitude,longitude,horizontal_accuracy
caid,,,,,
0010f2d60880f27d29454c77033005c588e94ce0205ba1b9dbe8149e132197cf,2024-03-25 01:27:00-07:00,idfa,33.597290,-117.675529,4
0010f2d60880f27d29454c77033005c588e94ce0205ba1b9dbe8149e132197cf,2024-03-25 01:27:01-07:00,idfa,33.597309,-117.675560,4
0010f2d60880f27d29454c77033005c588e94ce0205ba1b9dbe8149e132197cf,2024-03-25 01:27:02-07:00,idfa,33.597290,-117.675522,4
0010f2d60880f27d29454c77033005c588e94ce0205ba1b9dbe8149e132197cf,2024-03-25 01:27:04-07:00,idfa,33.597301,-117.675552,4
0010f2d60880f27d29454c77033005c588e94ce0205ba1b9dbe8149e132197cf,2024-03-25 01:27:05-07:00,idfa,33.586010,-117.671669,4
...,...,...,...,...,...
00117f0e6c1d23b451a314612f652ab828c7d277ab3ffda2596ab97343e9280f,2024-03-25 21:54:48-07:00,idfa,33.762383,-116.997063,14
00117f0e6c1d23b451a314612f652ab828c7d277ab3ffda2596ab97343e9280f,2024-03-25 23:17:05-07:00,idfa,33.762341,-116.997040,14
00117f0e6c1d23b451a314612f652ab828c7d277ab3ffda2596ab97343e9280f,2024-03-25 23:17:10-07:00,idfa,33.762352,-116.997063,14


In [15]:
## Pull multiple row groups
p.read_row_groups([15, 23, 34435]).to_pandas()

,datetime,id_type,latitude,longitude,horizontal_accuracy
caid,,,,,
0010f2d60880f27d29454c77033005c588e94ce0205ba1b9dbe8149e132197cf,2024-03-25 01:27:00-07:00,idfa,33.597290,-117.675529,4
0010f2d60880f27d29454c77033005c588e94ce0205ba1b9dbe8149e132197cf,2024-03-25 01:27:01-07:00,idfa,33.597309,-117.675560,4
0010f2d60880f27d29454c77033005c588e94ce0205ba1b9dbe8149e132197cf,2024-03-25 01:27:02-07:00,idfa,33.597290,-117.675522,4
0010f2d60880f27d29454c77033005c588e94ce0205ba1b9dbe8149e132197cf,2024-03-25 01:27:04-07:00,idfa,33.597301,-117.675552,4
0010f2d60880f27d29454c77033005c588e94ce0205ba1b9dbe8149e132197cf,2024-03-25 01:27:05-07:00,idfa,33.586010,-117.671669,4
...,...,...,...,...,...
99801c3f09c8419482df9dc1455a56eb110f58eff457c7c45ac21a1de433af6c,2024-03-25 02:45:10-07:00,idfa,34.528969,-117.315941,6
99801c3f09c8419482df9dc1455a56eb110f58eff457c7c45ac21a1de433af6c,2024-03-25 02:45:11-07:00,idfa,34.528980,-117.315910,6
99801c3f09c8419482df9dc1455a56eb110f58eff457c7c45ac21a1de433af6c,2024-03-25 02:45:18-07:00,idfa,34.528976,-117.315971,6


Since the same user IDs are mapped to the same numbered row groups across the entire dataset...we can efficiently pull user data across the entire time span without touching any other data using some simple multiprocessing.

Additionally, we'll load the metadata for each file first and keep it stored globally for access. The most time is spent opening and reading the metadata, so if we open it first and send it along it greatly speeds things up.

In [16]:
## load the metadata
meta = [pq.ParquetFile(x).metadata for x in daily_files]

Here's some help functions to pull the row groups from each file and compute a metric (the home location).

In [33]:
def get_row_groups(row_group, files):
    """ Read a specific row group across all files. """
    
    return pd.concat([pq.ParquetFile(f, metadata=m).read_row_group(row_group).to_pandas() for f, m in zip(files, meta)]) 


def get_row_group_trajectories(row_group, files):
    """ Read the row group from all files and return a skmobility Trajectory DataFrame """
    df = get_row_groups(row_group, files)
    tdf = TrajDataFrame(df.reset_index(), latitude='latitude', longitude='longitude', datetime='datetime', user_id='caid')
    
    return tdf

def get_home_location(row_group, files):
    """ Read the row group from all files and return the home location of each user. """

    tdf = get_row_group_trajectories(row_group, files)
    
    return individual.home_location(tdf, show_progress=False)

Here we can test it with a single row group (1000 users) to grab all the data from March and April.

In [24]:
%%time
row_group = 99
df = pd.concat([get_row_groups(99, daily_files)])
df

CPU times: user 53.3 ms, sys: 12.8 ms, total: 66 ms
Wall time: 63.8 ms


,datetime,id_type,latitude,longitude,horizontal_accuracy
caid,,,,,
007103d01df1d086de4ca1d17a470e45fbb53954c73c0b70a8e3886c7d017187,2024-03-14 18:36:32-07:00,idfa,34.078838,-118.240448,8
007103d01df1d086de4ca1d17a470e45fbb53954c73c0b70a8e3886c7d017187,2024-03-14 18:36:33-07:00,idfa,34.078850,-118.240410,8
007103d01df1d086de4ca1d17a470e45fbb53954c73c0b70a8e3886c7d017187,2024-03-14 18:36:34-07:00,idfa,34.078850,-118.240410,8
007103d01df1d086de4ca1d17a470e45fbb53954c73c0b70a8e3886c7d017187,2024-03-14 18:36:35-07:00,idfa,34.078823,-118.240486,8
007103d01df1d086de4ca1d17a470e45fbb53954c73c0b70a8e3886c7d017187,2024-03-14 18:36:36-07:00,idfa,34.078812,-118.240440,8
...,...,...,...,...,...
00714960a0c05f203e59b0b23167e735b80d96fbc14828cca08034a7d2353694,2024-04-30 08:56:01-07:00,aaid,33.888378,-118.308907,1
00714960a0c05f203e59b0b23167e735b80d96fbc14828cca08034a7d2353694,2024-04-30 08:56:07-07:00,aaid,33.888302,-118.308907,1
0071ad9e6eda0a3ea8be5aa80a43cddbf72db8cbfa248a01e53ac1b6aa170e6b,2024-04-30 06:05:00-07:00,aaid,36.747730,-119.772369,11


Wow, pretty fast! Let's verify that there are only a 1000 unique user IDs (one row group).

In [26]:
## yep, 1000 users...
df.index.nunique()

1000

Since we have more than one argument to our function, lets combine them to make it cleaner for the multiprocessing, so we can process a bunch more users.

In [29]:
row_groups = np.arange(0, 1000) # grab the first thousand row groups (1 million users)
files = [daily_files] * len(row_groups)
args = [(rg, f) for rg, f in zip(row_groups, files)]

In [30]:
%%time
with Pool(processes=8) as pool:

    data = pd.concat(pool.starmap(get_row_groups, args))
data

CPU times: user 7.25 s, sys: 7.7 s, total: 15 s
Wall time: 16.2 s


,datetime,id_type,latitude,longitude,horizontal_accuracy
caid,,,,,
0000332b899e1497d89f0011bb2057f5e0c6d3ac4cccf30e2a27427f729ded28,2024-03-14 22:59:00-07:00,idfa,33.726700,-118.000778,10
0000332b899e1497d89f0011bb2057f5e0c6d3ac4cccf30e2a27427f729ded28,2024-03-14 22:59:03-07:00,idfa,33.726700,-118.000771,10
0000332b899e1497d89f0011bb2057f5e0c6d3ac4cccf30e2a27427f729ded28,2024-03-14 22:59:10-07:00,idfa,33.726700,-118.000748,10
0000332b899e1497d89f0011bb2057f5e0c6d3ac4cccf30e2a27427f729ded28,2024-03-14 22:59:12-07:00,idfa,33.726761,-118.000763,10
0000332b899e1497d89f0011bb2057f5e0c6d3ac4cccf30e2a27427f729ded28,2024-03-14 22:59:18-07:00,idfa,33.726730,-118.000771,10
...,...,...,...,...,...
0473f636c2d52717249a170b9b7c6106c2416ff9c7cc7a847cfe2b4fd8eed0de,2024-04-30 11:45:50-07:00,aaid,34.161911,-116.433510,11
0473f636c2d52717249a170b9b7c6106c2416ff9c7cc7a847cfe2b4fd8eed0de,2024-04-30 11:45:56-07:00,aaid,34.161919,-116.433540,11
0473f636c2d52717249a170b9b7c6106c2416ff9c7cc7a847cfe2b4fd8eed0de,2024-04-30 11:45:59-07:00,aaid,34.161900,-116.433502,11


In [31]:
## yep, a million users
data.index.nunique()

1000000

Nice! A million users data fetched in 16 seconds. 

**Note**: Some of the parquet functions utilize multithreading, and some do not. So a balance of how many procesess to select is unclear to me and performance varies depending on the task.

Let's get the home location of these million users (from scrtach).

In [34]:
%%time
with Pool(processes=100) as pool:

    data = pd.concat(pool.starmap(get_home_location, args))
data

CPU times: user 571 ms, sys: 14.3 s, total: 14.8 s
Wall time: 56.1 s


,uid,lat,lng
0,0000001895b021c56bb407ea8c4743ed3eeea48272a3ef...,33.852242,-117.549667
1,000000aa47ce287ef4af544f71ee749f7b23ee1cd2c998...,41.736820,-122.624741
2,00000100fad94dbbd2bf0e74d6bdc4782c4e7207d6d9e3...,37.977982,-122.031067
3,0000019a8d62587a1063194cc9353fb814522e03b0e523...,34.214901,-119.176102
4,000001be5106d94ed20c1dfc8b126f45606196d0c5482d...,37.457699,-122.195778
...,...,...,...
995,0474dc71d5162a93cebc41745447f7e7d0930f948f1793...,33.955502,-118.286301
996,0474dc959b07d07ddc69daa6f1365cda4114e85acd66a7...,38.609600,-121.444298
997,0474dd65d7204ade8cf6229f3694ecd7138940fb9fca82...,38.635330,-121.455818
998,0474dd771ae2c6ab359e2aaf789e63b165968d3cd569d5...,34.105900,-117.585899


Less than a minute for a Million users home locations based on all of their location data. 

I imagine there are other optimizations to be had, but I think this will scale pretty well for what we're trying to do.